In [2]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [14]:
import json
import ollama
import numpy as np
from models.metric import CDE, exhaustive_CDE, EF, CDEF

## Load benchmark dataset

In [4]:
def parse_benchmark(path:str):
    
    with open(path) as f:
        dataset = json.load(f)
        dataset = dataset['Data']

    A = []
    B = []
    C = []

    for e in dataset:
        A.append(e['A'])
        B.append(e['B'])
        C.append(e['C'])

    return A,B,C

In [5]:
similarities_path = '../data/external/benchmark/test_similarities.json'
A_sim, B_sim, C_sim = parse_benchmark(similarities_path)

mix_path = '../data/external/benchmark/test_mix.json'
A_mix, B_mix, C_mix = parse_benchmark(mix_path)

types_path = '../data/external/benchmark/test_types.json'
A_type, B_type, C_type = parse_benchmark(types_path)

## Get text embeddings

In [6]:
EMBED_MODEL = 'nomic-embed-text'

def get_text_embedding_using_ollama(text:str):
    
    if text:
        response = ollama.embed(
            model=EMBED_MODEL,
            input=text,
        )

        return response["embeddings"]
    
    return None

In [7]:
def get_embeddings(entity_type:list):
    embed_type = []
    
    for entry in entity_type:
        new_entry = []
        for [entity, type] in entry:
            embed = get_text_embedding_using_ollama(entity)
            new_entry.append([embed, type])
        embed_type.append(new_entry)

    return embed_type

In [8]:
A_sim_embed = get_embeddings(A_sim)
B_sim_embed = get_embeddings(B_sim)
C_sim_embed = get_embeddings(C_sim)

A_type_embed = get_embeddings(A_type)
B_type_embed = get_embeddings(B_type)
C_type_embed = get_embeddings(C_type)

A_mix_embed = get_embeddings(A_mix)
B_mix_embed = get_embeddings(B_mix)
C_mix_embed = get_embeddings(C_mix)


## Test metric

In [9]:
def test_metric(metric_fun, A, B, C, beta = None, optimum_mini=True, verbose = False):

    n_passed = 0
    n_failed = 0
    ids_failed = []

    for i, (a, b, c) in enumerate(zip(A, B, C)):

        if beta is not None:
            s1 = metric_fun(a, a, beta)
            s2 = metric_fun(a, b, beta)
            s3 = metric_fun(a, c, beta)
        else:
            s1 = metric_fun(a, a)
            s2 = metric_fun(a, b)
            s3 = metric_fun(a, c)

        # if verbose: print(f'{a}\t{b}\t{c}')
        if optimum_mini:
            if s1 <= s2 <= s3:
                if verbose: print(f'TEST PASSED\t{s1}\t{s2}\t{s3}\n')
                n_passed += 1
            else:
                # if verbose: print(f'{a}\t{b}\t{c}')
                if verbose: print(f'TEST FAILED\t{s1}\t{s2}\t{s3}\n')
                n_failed += 1
                ids_failed.append(i)
        else:
            if s1 >= s2 >= s3:
                if verbose: print(f'TEST PASSED\t{s1}\t{s2}\t{s3}\n')
                n_passed += 1
            else:
                # if verbose: print(f'{a}\t{b}\t{c}')
                if verbose: print(f'TEST FAILED\t{s1}\t{s2}\t{s3}\n')
                n_failed += 1
                ids_failed.append(i)

    return n_passed, n_failed, ids_failed

In [26]:
p, f, id = test_metric(CDE, A_sim_embed, B_sim_embed, C_sim_embed, verbose=False)
print(f'SIM\tCDE\t\tPASSED: {p}\tFAILED: {f}\t\
IDS 0-39: {(np.array(id) <= 39).sum()}\t\
IDS 40-79: {((np.array(id) >= 40) & (np.array(id) <= 79)).sum()}\t\
IDS 80-119: {((np.array(id) >= 80) & (np.array(id) <= 119)).sum()}\t\
IDS 120-129: {(np.array(id) >= 120).sum()}\t\
')

p, f, id = test_metric(exhaustive_CDE, A_sim_embed, B_sim_embed, C_sim_embed, verbose=False)
print(f'SIM\texh_CDE\t\tPASSED: {p}\tFAILED: {f}\t\
IDS 0-39: {(np.array(id) <= 39).sum()}\t\
IDS 40-79: {((np.array(id) >= 40) & (np.array(id) <= 79)).sum()}\t\
IDS 80-119: {((np.array(id) >= 80) & (np.array(id) <= 119)).sum()}\t\
IDS 120-129: {(np.array(id) >= 120).sum()}\t\
')

p, f, id = test_metric(EF, A_sim_embed, B_sim_embed, C_sim_embed, verbose=False)
print(f'SIM\tEF\t\tPASSED: {p}\tFAILED: {f}\t\
IDS 0-39: {(np.array(id) <= 39).sum()}\t\
IDS 40-79: {((np.array(id) >= 40) & (np.array(id) <= 79)).sum()}\t\
IDS 80-119: {((np.array(id) >= 80) & (np.array(id) <= 119)).sum()}\t\
IDS 120-129: {(np.array(id) >= 120).sum()}\t\
')

for beta in range(0, 225, 25):
    beta = beta/100.0
    p, f, id= test_metric(CDEF, A_sim_embed, B_sim_embed, C_sim_embed, beta=beta, optimum_mini=False, verbose=False)
    print(f'SIM\tCDEF-{beta}\tPASSED: {p}\tFAILED: {f}\t\
IDS 0-39: {(np.array(id) <= 39).sum()}\t\
IDS 40-79: {((np.array(id) >= 40) & (np.array(id) <= 79)).sum()}\t\
IDS 80-119: {((np.array(id) >= 80) & (np.array(id) <= 119)).sum()}\t\
IDS 120-129: {(np.array(id) >= 120).sum()}\t\
')

SIM	CDE		PASSED: 112	FAILED: 18	IDS 0-39: 11	IDS 40-79: 4	IDS 80-119: 3	IDS 120-129: 0	
SIM	exh_CDE		PASSED: 66	FAILED: 64	IDS 0-39: 40	IDS 40-79: 6	IDS 80-119: 12	IDS 120-129: 6	
SIM	EF		PASSED: 81	FAILED: 49	IDS 0-39: 40	IDS 40-79: 0	IDS 80-119: 0	IDS 120-129: 9	
SIM	CDEF-0.0	PASSED: 110	FAILED: 20	IDS 0-39: 11	IDS 40-79: 4	IDS 80-119: 5	IDS 120-129: 0	
SIM	CDEF-0.25	PASSED: 119	FAILED: 11	IDS 0-39: 4	IDS 40-79: 4	IDS 80-119: 2	IDS 120-129: 1	
SIM	CDEF-0.5	PASSED: 120	FAILED: 10	IDS 0-39: 0	IDS 40-79: 4	IDS 80-119: 0	IDS 120-129: 6	
SIM	CDEF-0.75	PASSED: 117	FAILED: 13	IDS 0-39: 0	IDS 40-79: 4	IDS 80-119: 0	IDS 120-129: 9	
SIM	CDEF-1.0	PASSED: 117	FAILED: 13	IDS 0-39: 0	IDS 40-79: 4	IDS 80-119: 0	IDS 120-129: 9	
SIM	CDEF-1.25	PASSED: 117	FAILED: 13	IDS 0-39: 0	IDS 40-79: 4	IDS 80-119: 0	IDS 120-129: 9	
SIM	CDEF-1.5	PASSED: 117	FAILED: 13	IDS 0-39: 0	IDS 40-79: 4	IDS 80-119: 0	IDS 120-129: 9	
SIM	CDEF-1.75	PASSED: 117	FAILED: 13	IDS 0-39: 0	IDS 40-79: 4	IDS 80-119: 0	IDS 120-129: 9	
S

In [27]:
p, f, id= test_metric(CDE, A_type_embed, B_type_embed, C_type_embed, verbose=False)
print(f'TYPE\tCDE\t\tPASSED: {p}\tFAILED: {f}\t\
IDS 0-39: {(np.array(id) <= 39).sum()}\t\
IDS 40-79: {((np.array(id) >= 40) & (np.array(id) <= 79)).sum()}\t\
IDS 80-119: {((np.array(id) >= 80) & (np.array(id) <= 119)).sum()}\t\
IDS 120-129: {(np.array(id) >= 120).sum()}\t\
')

p, f, id= test_metric(exhaustive_CDE, A_type_embed, B_type_embed, C_type_embed, verbose=False)
print(f'TYPE\texh_CDE\t\tPASSED: {p}\tFAILED: {f}\t\
IDS 0-39: {(np.array(id) <= 39).sum()}\t\
IDS 40-79: {((np.array(id) >= 40) & (np.array(id) <= 79)).sum()}\t\
IDS 80-119: {((np.array(id) >= 80) & (np.array(id) <= 119)).sum()}\t\
IDS 120-129: {(np.array(id) >= 120).sum()}\t\
')

p, f, id= test_metric(EF, A_type_embed, B_type_embed, C_type_embed, verbose=False)
print(f'TYPE\tEF\t\tPASSED: {p}\tFAILED: {f}\t\
IDS 0-39: {(np.array(id) <= 39).sum()}\t\
IDS 40-79: {((np.array(id) >= 40) & (np.array(id) <= 79)).sum()}\t\
IDS 80-119: {((np.array(id) >= 80) & (np.array(id) <= 119)).sum()}\t\
IDS 120-129: {(np.array(id) >= 120).sum()}\t\
')

for beta in range(0, 225, 25):
    beta = beta/100.0
    p, f, id= test_metric(CDEF, A_type_embed, B_type_embed, C_type_embed, beta=beta, optimum_mini=False, verbose=False)
    print(f'TYPE\tCDEF-{beta}\tPASSED: {p}\tFAILED: {f}\t\
IDS 0-39: {(np.array(id) <= 39).sum()}\t\
IDS 40-79: {((np.array(id) >= 40) & (np.array(id) <= 79)).sum()}\t\
IDS 80-119: {((np.array(id) >= 80) & (np.array(id) <= 119)).sum()}\t\
IDS 120-129: {(np.array(id) >= 120).sum()}\t\
')

TYPE	CDE		PASSED: 130	FAILED: 0	IDS 0-39: 0	IDS 40-79: 0	IDS 80-119: 0	IDS 120-129: 0	
TYPE	exh_CDE		PASSED: 130	FAILED: 0	IDS 0-39: 0	IDS 40-79: 0	IDS 80-119: 0	IDS 120-129: 0	
TYPE	EF		PASSED: 81	FAILED: 49	IDS 0-39: 40	IDS 40-79: 0	IDS 80-119: 0	IDS 120-129: 9	
TYPE	CDEF-0.0	PASSED: 130	FAILED: 0	IDS 0-39: 0	IDS 40-79: 0	IDS 80-119: 0	IDS 120-129: 0	
TYPE	CDEF-0.25	PASSED: 124	FAILED: 6	IDS 0-39: 0	IDS 40-79: 0	IDS 80-119: 0	IDS 120-129: 6	
TYPE	CDEF-0.5	PASSED: 124	FAILED: 6	IDS 0-39: 0	IDS 40-79: 0	IDS 80-119: 0	IDS 120-129: 6	
TYPE	CDEF-0.75	PASSED: 124	FAILED: 6	IDS 0-39: 0	IDS 40-79: 0	IDS 80-119: 0	IDS 120-129: 6	
TYPE	CDEF-1.0	PASSED: 123	FAILED: 7	IDS 0-39: 0	IDS 40-79: 0	IDS 80-119: 0	IDS 120-129: 7	
TYPE	CDEF-1.25	PASSED: 121	FAILED: 9	IDS 0-39: 0	IDS 40-79: 0	IDS 80-119: 0	IDS 120-129: 9	
TYPE	CDEF-1.5	PASSED: 121	FAILED: 9	IDS 0-39: 0	IDS 40-79: 0	IDS 80-119: 0	IDS 120-129: 9	
TYPE	CDEF-1.75	PASSED: 121	FAILED: 9	IDS 0-39: 0	IDS 40-79: 0	IDS 80-119: 0	IDS 120-129: 9	
TYP

In [29]:
p, f, id= test_metric(CDE, A_mix_embed, B_mix_embed, C_mix_embed, verbose=False)
print(f'MIX\tCDE\t\tPASSED: {p}\tFAILED: {f}\t\
IDS 0-39: {(np.array(id) <= 39).sum()}\t\
IDS 40-79: {((np.array(id) >= 40) & (np.array(id) <= 79)).sum()}\t\
IDS 80-119: {((np.array(id) >= 80) & (np.array(id) <= 119)).sum()}\t\
IDS 120-129: {(np.array(id) >= 120).sum()}\t\
')

p, f, id= test_metric(exhaustive_CDE, A_mix_embed, B_mix_embed, C_mix_embed, verbose=False)
print(f'MIX\texh_CDE\t\tPASSED: {p}\tFAILED: {f}\t\
IDS 0-39: {(np.array(id) <= 39).sum()}\t\
IDS 40-79: {((np.array(id) >= 40) & (np.array(id) <= 79)).sum()}\t\
IDS 80-119: {((np.array(id) >= 80) & (np.array(id) <= 119)).sum()}\t\
IDS 120-129: {(np.array(id) >= 120).sum()}\t\
')

p, f, id= test_metric(EF, A_mix_embed, B_mix_embed, C_mix_embed, verbose=False)
print(f'MIX\tEF\t\tPASSED: {p}\tFAILED: {f}\t\
IDS 0-39: {(np.array(id) <= 39).sum()}\t\
IDS 40-79: {((np.array(id) >= 40) & (np.array(id) <= 79)).sum()}\t\
IDS 80-119: {((np.array(id) >= 80) & (np.array(id) <= 119)).sum()}\t\
IDS 120-129: {(np.array(id) >= 120).sum()}\t\
')

for beta in range(0, 225, 25):
    beta = beta/100.0
    p, f, id= test_metric(CDEF, A_mix_embed, B_mix_embed, C_mix_embed, beta=beta, optimum_mini=False, verbose=False)
    print(f'MIX\tCDEF-{beta}\tPASSED: {p}\tFAILED: {f}\t\
IDS 0-39: {(np.array(id) <= 39).sum()}\t\
IDS 40-79: {((np.array(id) >= 40) & (np.array(id) <= 79)).sum()}\t\
IDS 80-119: {((np.array(id) >= 80) & (np.array(id) <= 119)).sum()}\t\
IDS 120-129: {(np.array(id) >= 120).sum()}\t\
')

MIX	CDE		PASSED: 108	FAILED: 22	IDS 0-39: 18	IDS 40-79: 3	IDS 80-119: 1	IDS 120-129: 0	
MIX	exh_CDE		PASSED: 91	FAILED: 39	IDS 0-39: 25	IDS 40-79: 5	IDS 80-119: 6	IDS 120-129: 3	
MIX	EF		PASSED: 89	FAILED: 41	IDS 0-39: 31	IDS 40-79: 0	IDS 80-119: 4	IDS 120-129: 6	
MIX	CDEF-0.0	PASSED: 107	FAILED: 23	IDS 0-39: 18	IDS 40-79: 3	IDS 80-119: 2	IDS 120-129: 0	
MIX	CDEF-0.25	PASSED: 109	FAILED: 21	IDS 0-39: 17	IDS 40-79: 3	IDS 80-119: 1	IDS 120-129: 0	
MIX	CDEF-0.5	PASSED: 112	FAILED: 18	IDS 0-39: 15	IDS 40-79: 3	IDS 80-119: 0	IDS 120-129: 0	
MIX	CDEF-0.75	PASSED: 123	FAILED: 7	IDS 0-39: 4	IDS 40-79: 3	IDS 80-119: 0	IDS 120-129: 0	
MIX	CDEF-1.0	PASSED: 126	FAILED: 4	IDS 0-39: 0	IDS 40-79: 3	IDS 80-119: 0	IDS 120-129: 1	
MIX	CDEF-1.25	PASSED: 126	FAILED: 4	IDS 0-39: 0	IDS 40-79: 3	IDS 80-119: 0	IDS 120-129: 1	
MIX	CDEF-1.5	PASSED: 117	FAILED: 13	IDS 0-39: 0	IDS 40-79: 3	IDS 80-119: 4	IDS 120-129: 6	
MIX	CDEF-1.75	PASSED: 117	FAILED: 13	IDS 0-39: 0	IDS 40-79: 3	IDS 80-119: 4	IDS 120-129: 6	
MIX

## Debugging

In [13]:
from operator import itemgetter 

b = [90,92]
p, f, id= test_metric(
    CDEF, 
    itemgetter(*b)(A_sim_embed), 
    itemgetter(*b)(B_sim_embed), 
    itemgetter(*b)(C_sim_embed), 
    beta=0.0, optimum_mini=False, verbose=False)


print(f'SIM\tCDEF-0\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

SIM	CDEF-0	PASSED: 0	FAILED: 2	IDS FAIL: [0, 1]
